# 03 — Text Preprocessing + TF-IDF Baseline

Goals:
- Build and validate the text cleaning pipeline
- TF-IDF vectorization + Logistic Regression baseline
- TF-IDF + Random Forest
- Classification report per category
- Error analysis: which categories are most confused?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import sys; sys.path.insert(0, '..')
from src.nlp_pipeline import preprocess_series, build_tfidf_vectorizer, CATEGORY_LABELS

%matplotlib inline
sns.set_theme(style='darkgrid')

In [ ]:
df = pd.read_csv('../data/work_orders.csv')
df['clean_text'] = preprocess_series(df.text)

X_train, X_test, y_train, y_test = train_test_split(
    df.clean_text, df.failure_category, test_size=0.2, stratify=df.failure_category, random_state=42
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
# Baseline: TF-IDF + Logistic Regression
lr_pipeline = Pipeline([
    ('tfidf', build_tfidf_vectorizer(max_features=5000)),
    ('clf',   LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
print('=== TF-IDF + Logistic Regression ===')
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Random Forest
rf_pipeline = Pipeline([
    ('tfidf', build_tfidf_vectorizer(max_features=5000)),
    ('clf',   RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)),
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)
print('=== TF-IDF + Random Forest ===')
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Confusion matrix for LR (baseline)
cm = confusion_matrix(y_test, y_pred_lr, labels=CATEGORY_LABELS)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', ax=ax,
            xticklabels=[c.replace('_',' ')[:12] for c in CATEGORY_LABELS],
            yticklabels=[c.replace('_',' ')[:12] for c in CATEGORY_LABELS], cmap='Blues')
ax.set_title('Confusion Matrix — TF-IDF + LR Baseline')
plt.tight_layout()
plt.savefig('../figures/cm_tfidf_lr.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save best TF-IDF pipeline for API use
best = lr_pipeline  # update to rf if RF wins
joblib.dump(best, '../models/tfidf_pipeline.joblib')
print('Saved tfidf_pipeline.joblib')
print('Proceed to notebook 04 for DistilBERT fine-tuning.')